In [2]:
import os
os.environ['HSA_OVERRIDE_GFX_VERSION'] = '10.3.0'

import torch
print(torch.cuda.is_available())

True


# Input

In [3]:
raw_dummy = torch.tensor(
    [
        [
            -17.45557857, 2.174657688, 2.367182984, -16.99430584, -31.52132607,
        ],
        [
            -13.55773965, 1.965214473, -47.67187483, -5.831726978, -55.39959188,
        ],
        [
            -2.178412257, 9.753551178, -14.56978835, -76.58151401, -30.69740492,
        ],
        [
            -43.89739173, -50.37969167, -32.77935361, -66.49576026, -6.412497726,
        ],
        [
            -10.70635018, -72.22502322, -26.17916214, 1.4530419, 8.722773095,
        ],
        [
            -24.66762607, -18.16726717, 7.208516509, -13.64334883, -2.652998327,
        ],
    ]
)

In [4]:
raw_dummy_mean = torch.mean(raw_dummy, 0)
raw_dummy_std = torch.std(raw_dummy, 0, correction=0)

print(raw_dummy_mean)
print(raw_dummy_std)

tensor([-18.7439, -21.1464, -18.6041, -29.6823, -19.6602])
tensor([13.1362, 30.2911, 19.2597, 30.3075, 21.6416])


In [5]:
normal_dummy = (raw_dummy - raw_dummy_mean) / raw_dummy_std
print(normal_dummy)

tensor([[ 0.0981,  0.7699,  1.0889,  0.4186, -0.5481],
        [ 0.3948,  0.7630, -1.5093,  0.7870, -1.6514],
        [ 1.2611,  1.0201,  0.2095, -1.5474, -0.5100],
        [-1.9148, -0.9651, -0.7360, -1.2147,  0.6121],
        [ 0.6119, -1.6863, -0.3933,  1.0273,  1.3115],
        [-0.4509,  0.0984,  1.3402,  0.5292,  0.7859]])


# Frontend

In [6]:
import torch
import torch.nn as nn


class FrontEnd(nn.Module):
    def __init__(self):
        super(FrontEnd, self).__init__()
        # Vertical conv
        self.vert_conv = nn.Conv2d(
            in_channels=1, out_channels=1, kernel_size=(5, 3), padding="same"
        )
        self.vert_conv.weight = torch.nn.Parameter(
            torch.tensor(
                [
                    [-0.512, 0.897, -0.523],
                    [0.122, -0.204, -0.74],
                    [0.742, 0.805, -0.977],
                    [0.782, 0.516, -0.914],
                    [0.409, -0.438, 0.293],
                ]
            )
            .unsqueeze(0)
            .unsqueeze(0)
        )

        self.vert_conv.bias = torch.nn.Parameter(torch.ones(self.vert_conv.bias.shape))

        self.vert_pool = nn.MaxPool2d(kernel_size=(1, 5))

        # Horizontal conv
        self.mean_pool_input = nn.AvgPool2d(kernel_size=(1, 5))
        self.horz_conv = nn.Conv2d(
            in_channels=1, out_channels=1, kernel_size=(3, 1), padding="same"
        )
        self.horz_conv.weight = torch.nn.Parameter(
            torch.tensor(
                [
                    [0.122],
                    [0.516],
                    [-0.977],
                ]
            )
            .unsqueeze(0)
            .unsqueeze(0)
        )

        self.horz_conv.bias = torch.nn.Parameter(torch.ones(self.horz_conv.bias.shape))

        self.relu = nn.ReLU()

    def forward(self, x):
        # Vertical conv
        v = self.vert_conv(x)
        print("vert conv", v)
        v = self.relu(v)
        print("relu", v)
        v = self.vert_pool(v)  # Output shape: (Batch, 1, 6, 1)
        print("vert pool", v)

        # Horizontal conv
        h = self.mean_pool_input(x)
        print("mean pool input", h)
        h = self.horz_conv(h)
        print("horz conv", h)
        h = self.relu(h)  # Output shape: (Batch, 1, 6, 1)
        print("relu", h)

        # Concatenate along the feature dimension to get a 6x2 output
        out = torch.cat([v, h], dim=-1)  # Output shape: (Batch, 1, 6, 2)
        print("frontend output", out)
        return out

# CNN backend

In [7]:
class CNNBackEnd(nn.Module):
    def __init__(self):
        super(CNNBackEnd, self).__init__()
        self.conv = nn.Conv2d(
            in_channels=1, out_channels=1, kernel_size=(3, 2), padding="same"
        )

        self.conv.weight = torch.nn.Parameter(
            torch.tensor([[-0.204, -0.74], [0.805, -0.977], [0.516, -0.914]])
            .unsqueeze(0)
            .unsqueeze(0)
        )

        self.conv.bias = torch.nn.Parameter(torch.ones(self.conv.bias.shape))

        self.relu = nn.ReLU()

        # Pooling applied across the 2 features
        self.max_pool = nn.MaxPool2d(kernel_size=(1, 2))
        self.mean_pool = nn.AvgPool2d(kernel_size=(1, 2))

    def forward(self, x):
        print("\nCNN BACKEND")
        x = self.conv(x)
        print("cnn be conv", x)
        x = self.relu(x)
        print("cnn be relu", x)

        # Flatten pooling results to 1D vectors per batch
        p_max = self.max_pool(x).view(x.size(0), -1)  # 6 nodes
        p_mean = self.mean_pool(x).view(x.size(0), -1)  # 6 nodes

        # Concatenate to get 12 nodes total
        out = torch.cat([p_max, p_mean], dim=1)
        print("padding out to classifier", out)
        return out

# GRU backend

In [8]:
class GRUBackEnd(nn.Module):
    def __init__(self):
        super(GRUBackEnd, self).__init__()
        # input_size=2 (features), hidden_size=2
        self.gru = nn.GRU(input_size=2, hidden_size=2, num_layers=1, batch_first=True)

        # Order: reset, update, candidate
        custom_weight_ih = torch.tensor([
            [0.878, -0.139],
            [-0.013, 0.997],
            [-0.212, 0.248],
            [-0.253, -0.872],
            [-0.507, 0.891],
            [0.672, 0.741],
        ], dtype=torch.float32)

        custom_weight_hh = torch.tensor([
            [-0.253, -0.872],
            [-0.212, 0.248],
            [-0.013, 0.997],
            [0.878, -0.139],
            [0.672, 0.741],
            [-0.507, 0.891],
        ], dtype=torch.float32)

        self.gru.weight_ih_l0 = torch.nn.Parameter(custom_weight_ih)
        self.gru.weight_hh_l0 = torch.nn.Parameter(custom_weight_hh)
        self.gru.bias_ih_l0 = torch.nn.Parameter(torch.zeros(3 * 2))
        self.gru.bias_hh_l0 = torch.nn.Parameter(torch.zeros(3 * 2))

    def forward(self, x):
        print("\nGRU BACKEND")
        x = x.squeeze(
            1
        )  # Remove channel dim to match GRU sequence format: (Batch, 6, 2)
        out, _ = self.gru(x, torch.tensor([[0.805, -0.977]]).unsqueeze(0))
        print("gru out", out)

        # Pool across the hidden feature dimension
        p_max, _ = torch.max(out, dim=-1)  # 6 nodes
        p_mean = torch.mean(out, dim=-1)  # 6 nodes

        out = torch.cat([p_max, p_mean], dim=1)  # 12 nodes
        print("padding out to classifier", out)
        return out

# Attention backend

In [9]:
torch.arange(0, 2, 2)

tensor([0])

In [10]:
import math


class PositionalEncoding(nn.Module):
    def __init__(self, d_model=2, max_len=6):
        super(PositionalEncoding, self).__init__()

        pe = torch.zeros(max_len, d_model)
        N = 10000.0
        i = torch.arange(0, d_model // 2)

        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.pow(N, (2 * i) / d_model)
        # div_term = torch.exp(
        #     torch.arange(0, d_model, 2).float() * (-math.log(N) / d_model)
        # )

        pe[:, 0::2] = torch.sin(position / div_term)
        if d_model > 1:
            pe[:, 1::2] = torch.cos(position / div_term)

        self.pe = pe.unsqueeze(0)

    def forward(self, x):
        return x + self.pe[:, : x.size(1), :].to(x.device)


class AttentionBackEnd(nn.Module):
    def __init__(self):
        super(AttentionBackEnd, self).__init__()
        self.pos_encoder = PositionalEncoding(d_model=2)
        self.attention = nn.MultiheadAttention(
            embed_dim=2, num_heads=2, batch_first=True
        )

        self.ffn = nn.Sequential(nn.Linear(2, 2), nn.ReLU(), nn.Linear(2, 2))

        # QKV attention init weights
        wq = torch.tensor([[-0.862, 0.880], [-0.487, 0.238]], dtype=torch.float32)

        wk = torch.tensor([[0.409, -0.728], [0.251, -0.082]], dtype=torch.float32)

        wv = torch.tensor([[-0.052, -0.648], [0.945, 0.435]], dtype=torch.float32)

        self.attention.in_proj_weight = torch.nn.Parameter(
            torch.cat([wq, wk, wv], dim=0)
        )

        self.attention.in_proj_bias = torch.nn.Parameter(torch.zeros(6))

        w_out = torch.tensor([[-0.474, -0.068], [0.534, 0.788]], dtype=torch.float32).T
        self.attention.out_proj.weight = torch.nn.Parameter(w_out)

        self.attention.out_proj.bias = torch.nn.Parameter(torch.zeros(2))

        # FFN init weight
        self.ffn[0]._parameters["weight"] = torch.nn.Parameter(
            torch.tensor([[-0.474, 0.251], [0.945, 0.238]], dtype=torch.float32).T
        )
        self.ffn[0]._parameters["bias"] = torch.nn.Parameter(torch.ones(2))

        self.ffn[2]._parameters["weight"] = torch.nn.Parameter(
            torch.tensor([[-0.728, -0.862], [-0.648, 0.052]], dtype=torch.float32).T
        )
        self.ffn[2]._parameters["bias"] = torch.nn.Parameter(torch.ones(2))

    def forward(self, x):
        print("\nATTN BACKEND")
        x = x.squeeze(1)  # Shape: (Batch, 6, 2)
        x = self.pos_encoder(x)
        print("PE out", x)

        attn_out, _ = self.attention(x, x, x)
        res_attn = x + attn_out
        print("attn_out", attn_out)
        print("res_attn", res_attn)
        ffn_out = self.ffn(res_attn)
        res_ffn = res_attn + ffn_out
        print("ffn_out", ffn_out)
        print("res_ffn", res_ffn)

        p_max, _ = torch.max(res_ffn, dim=-1)
        p_mean = torch.mean(res_ffn, dim=-1)

        out = torch.cat([p_max, p_mean], dim=1)  # 12 nodes
        print("padding out to classifier", out)
        return out

# Classifier

In [11]:
class Classifier(nn.Module):
    def __init__(self):
        super(Classifier, self).__init__()
        # 12 input nodes mapped to 3 output themes
        self.fc = nn.Linear(in_features=12, out_features=3)

        self.fc.weight = torch.nn.Parameter(
            torch.tensor(
                [
                    [-0.134, -0.671, -0.778],
                    [-0.945, -0.04, 0.991],
                    [-0.62, 0.312, -0.121],
                    [-0.291, -0.202, 0.186],
                    [-0.577, -0.185, 0.173],
                    [0.572, 0.892, 0.757],
                    [-0.21, -0.863, 0.252],
                    [0.549, 0.185, -0.508],
                    [0.99, -0.947, -0.905],
                    [0.469, 0.365, -0.207],
                    [0.241, 0.486, -0.006],
                    [-0.167, -0.442, 0.01],
                ]
            ).T
        )

        self.fc.bias = torch.nn.Parameter(torch.ones(self.fc.bias.shape))

        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.fc(x)
        print("classifier in", x)
        x = self.sigmoid(x)
        print("classifier sigmoid", x)
        return x


class MusicLabeler(nn.Module):
    def __init__(self, backend_type="cnn"):
        """
        backend_type can be 'cnn', 'gru', or 'attention'
        """
        super(MusicLabeler, self).__init__()
        self.frontend = FrontEnd()

        if backend_type == "cnn":
            self.backend = CNNBackEnd()
        elif backend_type == "gru":
            self.backend = GRUBackEnd()
        elif backend_type == "attention":
            self.backend = AttentionBackEnd()

        self.classifier = Classifier()

    def forward(self, x):
        x = self.frontend(x)
        x = self.backend(x)
        out = self.classifier(x)
        return out


x = normal_dummy.detach().clone()
x = x.unsqueeze(0)

# Initialize models
model_cnn = MusicLabeler(backend_type="cnn")
model_gru = MusicLabeler(backend_type="gru")
model_attn = MusicLabeler(backend_type="attention")

# Perform feed-forward propagation
pred_cnn = model_cnn(x)
pred_gru = model_gru(x)
pred_attn = model_attn(x)

print("CNN+CNN predictions (gembira, sedih, tegang):", pred_cnn.detach().numpy())
print("CNN+GRU predictions (gembira, sedih, tegang):", pred_gru.detach().numpy())
print("CNN+Attn predictions (gembira, sedih, tegang):", pred_attn.detach().numpy())

vert conv tensor([[[-0.4203,  2.8409,  1.0095,  4.0297,  0.2232],
         [ 0.2569,  3.1758,  1.0363,  2.8219, -1.8208],
         [-0.8094,  3.1834,  3.3789, -0.5983, -1.6201],
         [ 1.4277,  0.7168, -2.9033,  1.5094, -0.0252],
         [ 4.5199, -0.3783,  0.1195, -0.6669,  3.6991],
         [ 0.4510,  0.6445,  1.2183, -0.6082,  3.0541]]],
       grad_fn=<SqueezeBackward1>)
relu tensor([[[0.0000, 2.8409, 1.0095, 4.0297, 0.2232],
         [0.2569, 3.1758, 1.0363, 2.8219, 0.0000],
         [0.0000, 3.1834, 3.3789, 0.0000, 0.0000],
         [1.4277, 0.7168, 0.0000, 1.5094, 0.0000],
         [4.5199, 0.0000, 0.1195, 0.0000, 3.6991],
         [0.4510, 0.6445, 1.2183, 0.0000, 3.0541]]], grad_fn=<ReluBackward0>)
vert pool tensor([[[4.0297],
         [3.1758],
         [3.3789],
         [1.5094],
         [4.5199],
         [3.0541]]], grad_fn=<MaxPool2DWithIndicesBackward0>)
mean pool input tensor([[[ 0.3655],
         [-0.2432],
         [ 0.0866],
         [-0.8437],
         [ 0.174

/home/hugoa/Documents/projects/skripsi-code/ai/venv-rocm/lib/python3.12/site-packages/torch/nn/modules/conv.py:548: UserWarning: Using padding='same' with even kernel lengths and odd dilation may require a zero-padded copy of the input be created (Triggered internally at /pytorch/aten/src/ATen/native/Convolution.cpp:1025.)
  return F.conv2d(


# Backpropagation

In [12]:
# Dummy target label (gembira, sedih, tegang)
y_true = torch.tensor([[1.0, 0.0, 1.0]], dtype=torch.float32)

# Dummy frequencies of the positive class in the training set (p_i)
p_frequencies = torch.tensor(
    [0.115384615, 0.615384615, 0.269230769], dtype=torch.float32
)


def bour_weighted_bce_loss(predictions, targets, p):
    """
    Weighted binary cross-entropy loss as defined in Bour (2021):
    l(x, y) = -(1/c) * sum( (2 / (1 + p_i)) * y_i * log(x_i) + (2p_i / (1 + p_i)) * (1 - y_i) * log(1 - x_i) )
    """
    C = predictions.size(-1)

    # Calculate weights for positive and negative classes
    weight_pos = 2.0 / (1.0 + p)
    weight_neg = (2.0 * p) / (1.0 + p)

    # Clamp predictions to avoid log(0) which results in NaN or Inf
    eps = 10e-12
    preds_clamped = torch.clamp(predictions, eps, 1.0 - eps)

    # Calculate the two terms of the loss
    term1 = weight_pos * targets * torch.log(preds_clamped)
    term2 = weight_neg * (1.0 - targets) * torch.log(1.0 - preds_clamped)

    # print(term1 + term2)

    # Sum over the classes and average over the batch and number of classes
    loss = -torch.sum(term1 + term2) / C
    return loss


def perform_manual_update_and_print(model_name, model, predictions, targets, p, lr=0.1):
    # Zero out any existing gradients
    model.zero_grad()

    # Calculate weighted BCE loss
    loss = bour_weighted_bce_loss(predictions, targets, p)

    print("\n" + "=" * 60)
    print(f"BACKPROPAGATION FOR: {model_name}")
    print(f"Loss: {loss.item()}")
    print("=" * 60)

    # Perform backward pass to calculate gradients
    loss.backward(retain_graph=True)

    # Iterate through parameters, print gradients, and print the manually updated weights
    for name, param in model.named_parameters():
        if param.requires_grad and param.grad is not None:
            print(f"\n--- Parameter: {name} ---")

            print("Gradient:")
            print(param.grad.detach().numpy())

            # weight update: W_new = W_old - learning_rate * Gradient
            updated_value = param.data - lr * param.grad

            print(f"Updated Value (Learning Rate = {lr}):")
            print(updated_value.detach().numpy())


learning_rate = 0.1

perform_manual_update_and_print(
    "CNN+CNN Model", model_cnn, pred_cnn, y_true, p_frequencies, lr=learning_rate
)
perform_manual_update_and_print(
    "CNN+GRU Model", model_gru, pred_gru, y_true, p_frequencies, lr=learning_rate
)
perform_manual_update_and_print(
    "CNN+Attention Model",
    model_attn,
    pred_attn,
    y_true,
    p_frequencies,
    lr=learning_rate,
)


BACKPROPAGATION FOR: CNN+CNN Model
Loss: 1.3943060636520386

--- Parameter: frontend.vert_conv.weight ---
Gradient:
[[[[-0.12846482 -0.01399608  0.28295845]
   [-0.02209081  0.4019405  -0.15361325]
   [ 0.22248617  0.45854515 -0.28813738]
   [ 0.02296555  0.18745926 -0.23327786]
   [ 0.0439674  -0.38474318 -0.4522169 ]]]]
Updated Value (Learning Rate = 0.1):
[[[[-0.49915355  0.89839965 -0.5512959 ]
   [ 0.12420908 -0.24419405 -0.7246387 ]
   [ 0.71975136  0.7591455  -0.9481863 ]
   [ 0.77970344  0.49725404 -0.8906722 ]
   [ 0.40460327 -0.39952567  0.3382217 ]]]]

--- Parameter: frontend.vert_conv.bias ---
Gradient:
[0.27393997]
Updated Value (Learning Rate = 0.1):
[0.972606]

--- Parameter: frontend.horz_conv.weight ---
Gradient:
[[[[ 0.05024188]
   [-0.37573895]
   [-0.48315105]]]]
Updated Value (Learning Rate = 0.1):
[[[[ 0.11697581]
   [ 0.55357385]
   [-0.9286849 ]]]]

--- Parameter: frontend.horz_conv.bias ---
Gradient:
[-0.3306461]
Updated Value (Learning Rate = 0.1):
[1.0330646